# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Read from Silver tables

In [0]:
crm_customers = spark.table("workspace.silver.crm_customers")
erp_cust      = spark.table("workspace.silver.erp_cust")
erp_loc       = spark.table("workspace.silver.erp_loc")

# Business Transformation — dim_customers

## Logika joinova

```
crm_customers (master)
    LEFT JOIN erp_cust  ON customer_id  ->  birthday_date, erp_gender
    LEFT JOIN erp_loc   ON customer_id  ->  country
```

CRM customer_id format: `NAS-123456`
ERP customer_id format: `123456`

Normalizujemo CRM id uklanjanjem `NAS-` da bi join radio.

In [0]:
# Korak 1: Normalizacija svih customer_id-jeva pre joina

# CRM: 'NAS-123456' -> '123456'
crm_clean = crm_customers.withColumn(
    "customer_id_normalized",
    F.regexp_replace(F.col("customer_id"), r'NAS-', '')
)

# ERP: Silver vec cisti, ali dodajemo kao odbrambeni sloj
erp_cust_clean = erp_cust.withColumn(
    "customer_id",
    F.regexp_replace(F.col("customer_id"), r'^NAS', '')
)

erp_loc_clean = erp_loc.withColumn(
    "customer_id",
    F.regexp_replace(F.col("customer_id"), r'^NAS', '')
)

In [0]:
# Korak 2: JOIN sve tri tabele
joined = (
    crm_clean
    .join(
        erp_cust_clean.select(
            F.col("customer_id").alias("erp_cust_id"),
            F.col("birthday_date"),
            F.col("gender").alias("erp_gender")
        ),
        crm_clean["customer_id_normalized"] == erp_cust_clean["customer_id"],
        how="left"
    )
    .join(
        erp_loc_clean.select(
            F.col("customer_id").alias("erp_loc_id"),
            F.col("country")
        ),
        crm_clean["customer_id_normalized"] == erp_loc_clean["customer_id"],
        how="left"
    )
)

In [0]:
# Korak 3: Finalna dim_customers
# ROW_NUMBER() -> surrogate key
# Gender logika: CRM primaran, ERP fallback (SCD Type 1 princip)

# Korak 1: Pripremi svaku tabelu — samo kolone koje trebaju, bez duplikata

crm_prep = crm_customers.select(
    F.regexp_replace(F.col("customer_id"), r'NAS-', '').alias("customer_id_normalized"),
    F.col("customer_id"),
    F.col("firstname"),
    F.col("lastname"),
    F.col("gender").alias("crm_gender"),
    F.col("marital_status"),
    F.col("create_date")
)

erp_cust_prep = erp_cust.select(
    F.col("customer_id").alias("erp_cust_key"),
    F.col("birthday_date"),
    F.col("gender").alias("erp_gender")
)

erp_loc_prep = erp_loc.select(
    F.col("customer_id").alias("erp_loc_key"),
    F.col("country")
)

# Korak 2: JOIN — sada nema duplikatnih kolona
joined = (
    crm_prep
    .join(erp_cust_prep, crm_prep["customer_id_normalized"] == erp_cust_prep["erp_cust_key"], how="left")
    .join(erp_loc_prep,  crm_prep["customer_id_normalized"] == erp_loc_prep["erp_loc_key"],  how="left")
)

# Korak 3: Finalna tabela — F.col() radi jer nema vise duplikata
window_spec = Window.orderBy("customer_id")

dim_customers = (
    joined
    .withColumn(
        "customer_key",
        F.row_number().over(window_spec)
    )
    .withColumn(
        "gender",
        F.when(
            (F.col("crm_gender").isNotNull()) & (F.col("crm_gender") != 'n/a'),
            F.col("crm_gender")
        ).otherwise(
            F.col("erp_gender")
        )
    )
    .select(
        F.col("customer_key"),
        F.col("customer_id"),
        F.col("firstname"),
        F.col("lastname"),
        F.col("gender"),
        F.col("marital_status"),
        F.col("birthday_date"),
        F.col("country"),
        F.col("create_date")
    )
)

dim_customers.display()

# Write to Gold table

In [0]:
%sql

drop table workspace.gold.dim_customers

In [0]:
(
    dim_customers.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.gold.dim_customers")
)

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10

In [0]:
%sql
SELECT country, COUNT(*) as broj_kupaca
FROM workspace.gold.dim_customers
GROUP BY country
ORDER BY broj_kupaca DESC

In [0]:
%sql
SELECT gender, COUNT(*) as broj
FROM workspace.gold.dim_customers
GROUP BY gender

# DESCRIBE HISTORY — Delta transaction log

Svaki write kreira novi commit u _delta_log/.
DESCRIBE HISTORY pokazuje taj log — osnova za Time Travel i audit.

In [0]:
%sql
DESCRIBE HISTORY workspace.gold.dim_customers

In [0]:
%sql
SELECT COUNT(*) as broj_redova_v0
FROM workspace.gold.dim_customers VERSION AS OF 0